# Week 4, The Data Cookbook: APIs and a Polite Scrape

**What you'll do today.** Two ways to get a corpus off the web: the **API** (the polite front
door) and a small, careful **scrape** (the back door, used with care).

In [ ]:
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, embeddings, labels, and
# charts are still there next week. (Outside Colab - e.g. the offline test harness -
# this falls back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
# First, install the few packages Colab doesn't already ship. If you opened this
# notebook with the whole repo, the line below uses our pinned versions:
%pip install -q -r requirements.txt -c constraints.txt

# Opened this notebook on its own, without the repo files? Comment the line above
# and use this explicit pinned install instead:
# %pip install -q requests beautifulsoup4 pandas

In [ ]:
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup
print("imports ok")

In [ ]:
# Are we running tiny + offline (the compatibility harness sets this), or in a real
# Colab class session? Everything below adapts so the notebook runs either way.
import os
SMOKE = os.environ.get("CULTURE_AS_DATA_SMOKE") == "1"
print("SMOKE mode (tiny, offline):", SMOKE)

## Route 1, the API (the polite front door)

An **endpoint** is a URL that returns *data* instead of a web page.

In [ ]:
AIC = "https://api.artic.edu/api/v1/artworks?limit=5&fields=id,title,date_display,artist_title"

# A tiny built-in sample so the notebook runs even with no network (the harness uses this).
SAMPLE = {"data": [
    {"id": 1, "title": "The Bedroom", "date_display": "1889", "artist_title": "Vincent van Gogh"},
    {"id": 2, "title": "A Sunday on La Grande Jatte", "date_display": "1884-86", "artist_title": "Georges Seurat"},
    {"id": 3, "title": "Nighthawks", "date_display": "1942", "artist_title": "Edward Hopper"},
]}

def get_json(url):
    if SMOKE:
        return SAMPLE
    try:
        r = requests.get(url, timeout=20, headers={"User-Agent": "culture-as-data-course"})
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print("network call failed, using sample:", e)
        return SAMPLE

payload = get_json(AIC)
df = pd.json_normalize(payload["data"])
print(df.to_string(index=False))

A documented URL became a table.

## Route 2, a polite scrape (the back door)

When there's no API, the AI writes a small `BeautifulSoup` scrape of a *simple* page.

In [ ]:
# We parse a built-in HTML string so the lesson never depends on a live site.
HTML = """
<html><body>
  <ul class="catalog">
    <li class="item"><span class="name">Blue Vase</span><span class="year">1901</span></li>
    <li class="item"><span class="name">Red Bowl</span><span class="year">1923</span></li>
    <li class="item"><span class="name">Green Jug</span><span class="year">1947</span></li>
  </ul>
</body></html>
"""
soup = BeautifulSoup(HTML, "html.parser")
rows = []
for li in soup.select("li.item"):
    rows.append({"name": li.select_one(".name").text, "year": li.select_one(".year").text})
scraped = pd.DataFrame(rows)
print(scraped.to_string(index=False))

## Reshape with the AI; you judge what came back

The AI wrangles columns, types, missing values.

In [ ]:
scraped["year"] = pd.to_numeric(scraped["year"], errors="coerce")
scraped["century"] = (scraped["year"] // 100 + 1).astype("Int64")
print(scraped.to_string(index=False))
print("\nIs this the corpus you wanted? Right rows, right columns, nothing silently dropped?")

In [ ]:
# Save your collected corpus to the Drive project folder so it is here next week.
import os
_out = os.path.join(PROJECT_DIR, "week04_corpus.csv")
scraped.to_csv(_out, index=False)
print("saved corpus ->", _out)

## You collected a corpus

You pulled data two ways and reshaped it.

**Sketch:** write your **Data Biography** (~400 words) and *actually collect your corpus*.